In [1]:
import os
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from scipy import sparse


In [ ]:

# =========================================================
# 0) 你的文件路径（按需修改）
# =========================================================
DATA_DIR = "./adatas/RRErythroid"  # 放 h5ad 的文件夹，比如 "./data"

# group_files = {
#     # ================= pre =================
#     "pre_res_Erythroid": os.path.join(DATA_DIR, "RR_pre_res_Erythroid.h5ad"),
#     "pre_nres_Erythroid": os.path.join(DATA_DIR, "RR_pre_nres_Erythroid.h5ad"),

#     "pre_res_LE": os.path.join(DATA_DIR, "RR_pre_res_LEunknown.h5ad"),
#     "pre_nres_LE": os.path.join(DATA_DIR, "RR_pre_nres_LEcDC2.h5ad"),

#     # ================= post =================
#     "post_res_Erythroid": os.path.join(DATA_DIR, "RR_post_res_Erythroid.h5ad"),
#     "post_nres_Erythroid": os.path.join(DATA_DIR, "RR_post_nres_Erythroid.h5ad"),

#     "post_res_LE": os.path.join(DATA_DIR, "RR_post_res_LE.h5ad"),
#     "post_nres_LE": os.path.join(DATA_DIR, "RR_post_nres_LE.h5ad"),
# }
group_files = {
    # ================= pre =================
    "HC_Erythroid": os.path.join(DATA_DIR, "RR_HC_Erythroid.h5ad"),
    "post_nres_Erythroid": os.path.join(DATA_DIR, "RR_post_nres_Erythroid.h5ad"),
    "pre_res_Erythroid": os.path.join(DATA_DIR, "RR_pre_res_Erythroid.h5ad"),
    "pre_nres_Erythroid": os.path.join(DATA_DIR, "RR_pre_nres_Erythroid.h5ad"),

}
# 你要比较的 clusters（最终会分别输出）
# 每个条目：(cluster_name, res_key, nres_key)
# clusters_to_test = [
#     ("pre_res",  "pre_res_Erythroid",  "pre_res_LE"),
#     ("pre_nres", "pre_nres_Erythroid", "pre_nres_LE"),
#     ("post_res", "post_res_Erythroid", "post_res_LE"),
#     ("post_nres","post_nres_Erythroid","post_nres_LE"),
# ]
# clusters_to_test = [
#     ("pre_res",  "pre_res_LE",  "pre_res_Erythroid"),
#     ("pre_nres", "pre_nres_LE", "pre_nres_Erythroid"),
#     ("post_res", "post_res_LE", "post_res_Erythroid"),
#     ("post_nres","post_nres_LE","post_nres_Erythroid"),
# ]
clusters_to_test = [
    ("HCvspost_nres",  "HC_Erythroid",  "post_nres_Erythroid"),
    ("HCvspre_nres",  "HC_Erythroid",  "pre_nres_Erythroid"),
    ("HCvspre_res",  "HC_Erythroid",  "pre_res_Erythroid"),
]

# =========================================================
# 1) 参数（跟你之前逻辑一致）
# =========================================================
USE_LAYER = None       # e.g. "counts" / "lognorm"；不用就 None
USE_RAW = False        # 若要用 adata.raw，就 True（优先于 layers）
THRESHOLD = 0.0       # positive-only：只用 >threshold 的表达细胞
MIN_POS_CELLS = 20     # 每组至少多少表达细胞才纳入
USE_FDR = True         # 建议 True：多重检验
P_CUTOFF = 0.05
FDR_CUTOFF = 0.05
FC_METHOD = "mean"     # "mean" 或 "median"
FC_PSEUDOCOUNT = 0.0   # 如担心除零，可设 1e-9

# 是否先做 normalize/log1p（强烈建议 True，除非你的 X 已经是 lognorm）
DO_PREPROCESS = False
TARGET_SUM = 1e4

# 输出目录
OUTDIR = "DE_pre_nres_vs_pre_res_by_cluster"
os.makedirs(OUTDIR, exist_ok=True)

# =========================================================
# 2) 工具函数
# =========================================================
def bh_fdr(pvals):
    """Benjamini–Hochberg FDR；返回与输入同长度的 qvals（np.array）"""
    pvals = np.asarray(pvals, dtype=float)
    qvals = np.full_like(pvals, np.nan, dtype=float)
    ok = np.isfinite(pvals)
    if ok.sum() == 0:
        return qvals
    pv = pvals[ok]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    q = ranked * m / (np.arange(1, m + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = q
    qvals[ok] = out
    return qvals

def get_matrix_and_genes(adata, use_layer=None, use_raw=False):
    """
    返回 (X, gene_names)
    X: cells x genes (csr_matrix or ndarray)
    gene_names: np.array of gene symbols
    """
    if use_raw:
        if adata.raw is None:
            return None, None
        X = adata.raw.X
        genes = np.asarray(adata.raw.var_names)
    elif use_layer is not None:
        if use_layer not in adata.layers:
            return None, None
        X = adata.layers[use_layer]
        genes = np.asarray(adata.var_names)
    else:
        X = adata.X
        genes = np.asarray(adata.var_names)

    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = np.asarray(X)
    return X, genes

def summary_stat(x, method="mean"):
    if method == "median":
        return float(np.median(x))
    return float(np.mean(x))

def conditional_mw_positive_only_allgenes_okonly(
    ad_res, ad_nres, res_name, nres_name,
    min_pos_cells=20, threshold=0.0,
    use_layer=None, use_raw=False,
    fc_method="mean",
    fc_pseudocount=0.0
):
    """
    对所有共有基因做 positive-only MWU：
      - 仅使用 >threshold 的细胞
      - 若任一组 positive cells < min_pos_cells：丢弃该基因
      - 计算 FC（nres/res）和 log2FC（nres/res），基于 positive-only 且用 mean/median
    """
    X1, genes1 = get_matrix_and_genes(ad_res, use_layer=use_layer, use_raw=use_raw)
    X2, genes2 = get_matrix_and_genes(ad_nres, use_layer=use_layer, use_raw=use_raw)
    if X1 is None or X2 is None:
        return pd.DataFrame()

    common = np.intersect1d(genes1, genes2, assume_unique=False)
    if common.size == 0:
        return pd.DataFrame()

    idx1 = pd.Index(genes1).get_indexer(common)
    idx2 = pd.Index(genes2).get_indexer(common)

    X1c = X1[:, idx1]
    X2c = X2[:, idx2]

    rows = []
    for j, gene in enumerate(common):
        if sparse.issparse(X1c):
            x1_all = X1c[:, j].toarray().ravel()
            x2_all = X2c[:, j].toarray().ravel()
        else:
            x1_all = np.asarray(X1c[:, j]).ravel()
            x2_all = np.asarray(X2c[:, j]).ravel()

        x1_pos = x1_all[x1_all > threshold]  # res
        x2_pos = x2_all[x2_all > threshold]  # nres

        if (x1_pos.size < min_pos_cells) or (x2_pos.size < min_pos_cells):
            continue

        u, p = mannwhitneyu(x1_pos, x2_pos, alternative="two-sided")

        stat_res  = summary_stat(x1_pos, method=fc_method)
        stat_nres = summary_stat(x2_pos, method=fc_method)

        denom = stat_res + fc_pseudocount
        numer = stat_nres + fc_pseudocount
        fc = (numer / denom) if denom > 0 else np.nan
        log2fc = np.log2(fc) if (np.isfinite(fc) and fc > 0) else np.nan

        rows.append({
            "gene": gene,
            "p_value": float(p),
            "U": float(u),
            f"{res_name}_n_pos": int(x1_pos.size),
            f"{nres_name}_n_pos": int(x2_pos.size),
            f"{fc_method}_g1_pos": float(stat_res),
            f"{fc_method}_g2_pos": float(stat_nres),
            f"FC_g2_over_g1": float(fc) if np.isfinite(fc) else np.nan,
            f"log2FC_g2_over_g1": float(log2fc) if np.isfinite(log2fc) else np.nan,
            # f"median_pos_diff({nres_name}-{res_name})": float(np.median(x1_pos) - np.median(x2_pos)),
            f"mean_pos_diff(g1-g2)": float(np.mean(x1_pos) - np.mean(x2_pos)),
        })

    return pd.DataFrame(rows)

def maybe_preprocess(adata: sc.AnnData) -> sc.AnnData:
    """
    可选：对每个 adata 做 normalize_total + log1p（在 copy 上做）
    若你的 X 已经是 lognorm（比如 Scanpy 标准流程），就把 DO_PREPROCESS 设为 False
    """
    if not DO_PREPROCESS:
        return adata
    ad = adata.copy()
    sc.pp.normalize_total(ad, target_sum=TARGET_SUM)
    sc.pp.log1p(ad)
    return ad

# =========================================================
# 3) 读取数据
# =========================================================
adatas = {}
for k, fp in group_files.items():
    if not os.path.exists(fp):
        raise FileNotFoundError(f"Missing file for {k}: {fp}")
    adatas[k] = sc.read_h5ad(fp)

# =========================================================
# 4) 主流程：逐 cluster 跑 DE（pre_nres vs pre_res）
# =========================================================
all_sig_tables = []
log2fc_by_cluster = {}   # cluster -> pd.Series(gene -> log2FC)
sig_gene_sets = {}       # cluster -> set(sig_up_genes)

for cluster_name, res_key, nres_key in clusters_to_test:
    ad_res = maybe_preprocess(adatas[res_key])
    ad_nres = maybe_preprocess(adatas[nres_key])

    df = conditional_mw_positive_only_allgenes_okonly(
        ad_res, ad_nres,
        res_name=res_key, nres_name=nres_key,
        min_pos_cells=MIN_POS_CELLS,
        threshold=THRESHOLD,
        use_layer=USE_LAYER,
        use_raw=USE_RAW,
        fc_method=FC_METHOD,
        fc_pseudocount=FC_PSEUDOCOUNT
    )

    if df.shape[0] == 0:
        print(f"[WARN] {cluster_name}: no genes passed MIN_POS_CELLS.")
        continue

    df["FDR_BH"] = bh_fdr(df["p_value"].values)

    # 显著性 + 上调(nres>res)
    if USE_FDR:
        sig_mask = df["FDR_BH"].notna() & (df["FDR_BH"] < FDR_CUTOFF)
    else:
        sig_mask = df["p_value"].notna() & (df["p_value"] < P_CUTOFF)

    up_mask = df[f"log2FC_g2_over_g1"].notna() & (df[f"log2FC_g2_over_g1"] > 0)
    sig_up = df[sig_mask & up_mask].copy()

    # 排序：更靠前=更显著
    sig_up = sig_up.sort_values(["FDR_BH", "p_value"], ascending=True)

    # 保存
    out_full = os.path.join(OUTDIR, f"{cluster_name}__full_all_ok_genes.csv")
    out_sig  = os.path.join(OUTDIR, f"{cluster_name}__SIG_up_in_{nres_key}.csv")
    df.to_csv(out_full, index=False)
    sig_up.to_csv(out_sig, index=False)

    print(f"[{cluster_name}] ok_genes={df.shape[0]}  sig_up={sig_up.shape[0]}")
    print(f"  saved: {out_sig}")

    # 汇总用
    sig_up["cluster"] = cluster_name
    all_sig_tables.append(sig_up)

    log2fc_by_cluster[cluster_name] = df.set_index("gene")[f"log2FC_g2_over_g1"]
    sig_gene_sets[cluster_name] = set(sig_up["gene"].tolist())

# 合并输出：所有 cluster 的显著上调基因清单（你要的 “comprehensively list”）
if len(all_sig_tables) > 0:
    all_sig_df = pd.concat(all_sig_tables, ignore_index=True)
    all_sig_df.to_csv(os.path.join(OUTDIR, f"ALL_clusters__SIG_up_in_{nres_key}__combined.csv"), index=False)

    # 也给一个更“列表式”的版本：每个 cluster 一行，genes 用分号拼起来
    list_rows = []
    for cluster_name, genes in sig_gene_sets.items():
        list_rows.append({
            "cluster": cluster_name,
            "n_sig_up_genes": len(genes),
            "genes_sig_up_in_pre_nres": ";".join(sorted(list(genes)))
        })
    pd.DataFrame(list_rows).to_csv(
        os.path.join(OUTDIR, f"ALL_clusters__SIG_up_in_{nres_key}__gene_list_per_cluster.csv"),
        index=False
    )

# # =========================================================
# # 5) 画图：跨所有 cluster，突出 LSC 中 FC>1.3 的基因
# # =========================================================
# # 取 LSC 中：显著上调 且 FC>1.3
# LSC_NAME = "LSC"
# if LSC_NAME in log2fc_by_cluster:
#     # 先拿 LSC 的“显著上调表”文件读回来（最稳）
#     lsc_sig_path = os.path.join(OUTDIR, f"{LSC_NAME}__SIG_up_in_pre_nres.csv")
#     if os.path.exists(lsc_sig_path):
#         lsc_sig = pd.read_csv(lsc_sig_path)
#         lsc_sig_fc13 = lsc_sig[lsc_sig["FC_nres_over_res"].notna() & (lsc_sig["FC_nres_over_res"] > 1.3)].copy()

#         genes_focus = lsc_sig_fc13["gene"].dropna().unique().tolist()
#         print(f"[Plot] LSC significant & FC>1.3 genes: {len(genes_focus)}")

#         if len(genes_focus) > 0:
#             # 组一个矩阵：rows=genes_focus, cols=clusters, values=log2FC
#             clusters_order = [c[0] for c in clusters_to_test if c[0] in log2fc_by_cluster]
#             mat = []
#             for g in genes_focus:
#                 row = []
#                 for c in clusters_order:
#                     s = log2fc_by_cluster[c]
#                     row.append(float(s.get(g, np.nan)))
#                 mat.append(row)
#             mat = np.array(mat, dtype=float)

#             # 简单排序：按 LSC 的 log2FC 从大到小
#             if LSC_NAME in clusters_order:
#                 lsc_col = clusters_order.index(LSC_NAME)
#                 order = np.argsort(-np.nan_to_num(mat[:, lsc_col], nan=-1e9))
#                 mat = mat[order, :]
#                 genes_focus = [genes_focus[i] for i in order]

#             plt.figure(figsize=(1.2*len(clusters_order) + 4, 0.18*len(genes_focus) + 4))
#             im = plt.imshow(mat, aspect="auto")  # 默认 colormap
#             plt.colorbar(im, label="log2FC (pre_nres / pre_res)")

#             plt.xticks(range(len(clusters_order)), clusters_order, rotation=30, ha="right")
#             plt.yticks(range(len(genes_focus)), genes_focus, fontsize=7)

#             plt.title("Genes significant in LSC with FC>1.3: log2FC across clusters")
#             plt.tight_layout()

#             fig_path = os.path.join(OUTDIR, "HEATMAP__LSC_sig_FCgt1p3_genes_across_clusters.png")
#             plt.savefig(fig_path, dpi=300)
#             plt.close()
#             print(f"[Plot] saved: {fig_path}")

#             # 也输出该图对应矩阵
#             heat_df = pd.DataFrame(mat, index=genes_focus, columns=clusters_order)
#             heat_df.to_csv(os.path.join(OUTDIR, "HEATMAP__LSC_sig_FCgt1p3_genes_across_clusters__log2FC_matrix.csv"))
# else:
#     print("[WARN] LSC not found in results; skip heatmap.")

print("Done.")
